# 📓 Semana 5 · Dia 1 — Auto Loader: ingestão incremental

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA, DEP (ingestão) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline Auto Loader rodando com arquivos novos |

---


## 📖 Teoria — O problema da ingestão

Arquivos novos chegam o tempo todo (vendas diárias, logs). Reler tudo a cada vez é caro; perder arquivos é inaceitável.

**Auto Loader** (padrão Databricks para ingestão) monitora uma pasta e processa **somente arquivos novos**, com garantia **exactly-once** (checkpoint guarda o que já foi processado).


## 📖 Teoria — Como funciona

1. Você aponta para um diretório (`/Volumes/.../landing`).
2. O Auto Loader registra cada arquivo novo (via listagem ou notificação de arquivo — na Free, listagem).
3. Opções: `cloudFiles.format`, `cloudFiles.schemaLocation` (checkpoint de schema), `cloudFiles.inferColumnTypes`.

```python
(spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('cloudFiles.schemaLocation', '/Volumes/.../checkpoints')
    .load('/Volumes/.../landing'))
```


### 💻 Na prática — Preparando a área de landing

Crie a pasta de arquivos e um volume de trabalho.


In [ ]:
# Área de staging
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.vol_landing")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.vol_checkpoints")
print("Volumes de landing e checkpoints prontos")

In [ ]:
# Copiar uma amostra do dataset para o landing
import shutil
amostra = spark.table("workspace.bronze.vendas_bronze").limit(2000)
amostra.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/bronze/vol_landing/vendas")
print("2.000 linhas gravadas como CSV no landing")

### 💻 Na prática — Auto Loader na prática

Leia incrementalmente os arquivos novos do landing.


In [ ]:
# Leitura incremental com Auto Loader
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_landing")
    .option("header", True)
    .option("inferColumnTypes", True)
    .load("/Volumes/workspace/bronze/vol_landing"))
print("Stream configurado (lazy). Tipo:", type(df_stream).__name__)

In [ ]:
# Gravar incrementalmente (append) com checkpoint
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/bronze/vol_checkpoints/ckpt_landing")
    .outputMode("append")
    .trigger(once=True)   # roda uma vez, processa o que há
    .table("workspace.bronze.vendas_landing"))
query.awaitTermination()
print("Ingestão concluída:", spark.table("workspace.bronze.vendas_landing").count(), "linhas")

### 💻 Na prática — Novos arquivos

Adicione mais dados e rode de novo — só o que é novo entra (exactly-once).


In [ ]:
# Adicionar mais 1000 linhas novas ao landing
extra = spark.table("workspace.bronze.vendas_bronze").limit(1000)
extra.write.mode("append").option("header", True).csv("/Volumes/workspace/bronze/vol_landing/vendas_extra")
print("Mais 1.000 linhas no landing")

In [ ]:
# Rodar a ingestão de novo — só processa o que é novo
q2 = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_landing")
    .option("header", True)
    .load("/Volumes/workspace/bronze/vol_landing")
    .writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/bronze/vol_checkpoints/ckpt_landing")
    .outputMode("append")
    .trigger(once=True)
    .table("workspace.bronze.vendas_landing"))
q2.awaitTermination()
print("Agora total:", spark.table("workspace.bronze.vendas_landing").count(), "linhas (2000 + 1000 novas)")

> 🎯 **Dica de prova**: Auto Loader cai na DEA e DEP: `cloudFiles.format`, `schemaLocation`, exactly-once via checkpoint. Pergunta típica: 'qual opção para processar somente arquivos novos?' → Auto Loader.


## 🎯 Exercícios de fixação

**1.** Por que o Auto Loader é melhor que reler a pasta inteira?

**2.** O que `cloudFiles.schemaLocation` guarda?

**3.** Adicione um terceiro arquivo e verifique se o total continua correto.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Incremental

Processa só o delta (arquivos novos), com checkpoint — economiza custo e tempo, com garantia exactly-once.

**2.** schemaLocation

O checkpoint de schema: onde o Auto Loader persiste o schema inferido e a evolução — necessário para idempotência.

**3.** Terceiro arquivo

Total = 2000 + 1000 + novos; rode o stream mais uma vez e confira o count.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*